In [25]:
from __future__ import annotations

from pathlib import Path
import os

import numpy as np
import pandas as pd
import pydicom
import torch
from skimage.transform import resize
from torch.utils.data import DataLoader, Dataset


def ct_normalization(img, meta):
    """
    Normaliza el rango HU a [0, 1].
    Ajustamos de -1024 (aire) a 3072 (tejido denso) para cubrir un rango amplio
    para resaltar detalles anatómicos.
    """
    def scale(img, meta, slope=1.0, intersept=-1024):

        slope = float(meta.get("Rescale Slope", slope))
        intersept = float(meta.get("Rescale Intercept", intersept))
        
        img = img * slope + intersept  # Aplica la transformación de escala y desplazamiento
        img = resize(img, (256, 256), preserve_range=True)  # Asegura que todas las imágenes tengan el mismo tamaño
        img = (img + 1024) / (3072 + 1024)  # Escala a [0, 1]
        return img.astype(np.float32)
    
    return scale(img, meta)


def _read_dicom(path: Path) -> np.ndarray:
    
    ds = pydicom.dcmread(path)  # Lee el archivo DICOM y convierte a numpy array
    metadata = {}
    for element in ds:
            if getattr(element, "name", None) == "Pixel Data":
                continue
            try:
                name = str(element.name)
                value = element.value
                if value is None:
                    continue
                metadata[name] = str(value)
            except Exception:
                continue

    return ct_normalization(ds.pixel_array, metadata)


class DatasetLoader(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        dim: int = 2,
        training: str = "model",
    ) -> None:
        """
        Args:
            df: DataFrame with either:
                - dim == 2: columns ['Case', 'SDCT_file', 'LDCT_file'] or ['Case', 'SDCT', 'LDCT']
                - dim == 3: columns ['Case', 'SDCT', 'LDCT'] where SDCT/LDCT are volume directories
            transform: Optional transform to be applied on a sample.
            training: "model" or "vae", used to determine how to process the data.
        """
        self.dim = dim
        self.df = df
        self.training = training

        self._build_index()
        self._preprocess()
        # self._tensor_loader()

    def _build_index(self) -> pd.DataFrame:
        indexed = self.df.reset_index(drop=True)   
        indexed.insert(0, 'idx', indexed.index) 
        self.df = indexed
    
    def _preprocess(self):
        self.df = self.load_slices()

        aux = lambda dim, modalities: self.df.groupby('Case').agg({
            modality: list for modality in modalities
        }).reset_index() if dim == 3 else self.df

        if self.training == "model":
            aux(self.dim, ["SDCT", "LDCT"])
            self.df = self.df.dropna(subset=['SDCT', 'LDCT'])
        elif self.training == "vae":
            aux(self.dim, ["SDCT"])
            self.df = self.df.dropna(subset=['SDCT'])

    def _tensor_loader(self, idx):
        if self.training == "model":
            modalities = ["SDCT", "LDCT"]
        elif self.training == "vae":
            modalities = ["SDCT"]
        for idx in range(len(self.df)):
            self.load_dicom(idx, modalities=modalities)

    def load_slices(self) -> pd.DataFrame:
        rows = []
        for row in self.df.itertuples(index=False):
            sdct_dir = Path(row.SDCT)
            sdct_files = sorted(sdct_dir.glob("*.dcm"))

            if self.training == "model":
                ldct_files = sorted(Path(row.LDCT).glob("*.dcm"))
                rows.extend([
                    {'SDCT': s, 'LDCT': l, 'Case': row.Case} 
                    for s, l in zip(sdct_files, ldct_files)
                ])
            elif self.training == "vae":
                rows.extend([{'SDCT': s, 'Case': row.Case} for s in sdct_files])
                
        return pd.DataFrame(rows)
    
    def load_dicom(self, idx: int, modalities: list[str] = ["SDCT", "LDCT"]) -> dict:
        row = self.df.loc["idx", idx]
        if self.dim == 3:
            for modality in modalities:
                volume = np.array([_read_dicom(f) for f in sorted(row[modality])]) # Shape: (D, H, W)
                tensor = torch.from_numpy(volume).unsqueeze(0).float()  # Shape: (1, D, H, W)
                return tensor
                self.df.iloc[idx][modality] = tensor  # Cache the loaded tensor in the DataFrame
        else:
            for modality in modalities:
                img = _read_dicom(Path(row[modality]))
                tensor = torch.from_numpy(img).unsqueeze(0).float()  # Shape: (1, H, W)
                return tensor
                self.df.iloc[idx][modality] = tensor

    def __len__(self) -> int:
        return len(self.df)
    
    def __getitem__(self, idx: int) -> dict:
        row = self.df.loc["idx", idx]
        sample = {'Case': row['Case']}
        if self.training == "model":
            # sample['SDCT'] = row['SDCT']
            sample['SDCT'] = self.load_dicom(idx, modalities=['SDCT'])
            # sample['LDCT'] = row['LDCT']
            sample['LDCT'] = self.load_dicom(idx, modalities=['LDCT'])
        elif self.training == "vae":
            # sample['SDCT'] = row['SDCT']
            sample['SDCT'] = self.load_dicom(idx, modalities=['SDCT'])
        
        return sample

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [26]:
train = False
dataset = pd.read_csv(os.path.join("Dataset", 'train.txt' if train else 'test.txt'), sep='\t', names=('Case','SDCT','LDCT',))
print("Dataset loaded successfully.")
dataset = DatasetLoader(dataset, dim=3, training="vae")
print("Dataset object created successfully.")

Dataset loaded successfully.
Dataset object created successfully.


In [11]:
loader = DataLoader(dataset.df, batch_size=1, shuffle=True)
loader = iter(loader)

In [14]:
dataset.__getitem__(0)

{'Case': 'C012',
 'SDCT': [PosixPath('/home/shared_data/LDCT/manifest-1648648375084/LDCT-and-Projection-data/C012/12-23-2021-NA-NA-93425/1.000000-Full Dose Images-93580/1-001.dcm'),
  PosixPath('/home/shared_data/LDCT/manifest-1648648375084/LDCT-and-Projection-data/C012/12-23-2021-NA-NA-93425/1.000000-Full Dose Images-93580/1-002.dcm'),
  PosixPath('/home/shared_data/LDCT/manifest-1648648375084/LDCT-and-Projection-data/C012/12-23-2021-NA-NA-93425/1.000000-Full Dose Images-93580/1-003.dcm'),
  PosixPath('/home/shared_data/LDCT/manifest-1648648375084/LDCT-and-Projection-data/C012/12-23-2021-NA-NA-93425/1.000000-Full Dose Images-93580/1-004.dcm'),
  PosixPath('/home/shared_data/LDCT/manifest-1648648375084/LDCT-and-Projection-data/C012/12-23-2021-NA-NA-93425/1.000000-Full Dose Images-93580/1-005.dcm'),
  PosixPath('/home/shared_data/LDCT/manifest-1648648375084/LDCT-and-Projection-data/C012/12-23-2021-NA-NA-93425/1.000000-Full Dose Images-93580/1-006.dcm'),
  PosixPath('/home/shared_data/LD

In [13]:
batch = next(loader)
while batch is not None:
    print(batch['Case'])
    batch = next(loader, None)

KeyError: 23

In [ ]:
import torch
import torch.nn as nn
from monai.networks.nets import AutoencoderKL
import matplotlib.pyplot as plt




# --- 1. Definición de Arquitectura compatible con MedVAE ---
# Nota: MedVAE suele usar 128 canales base y 3 o 4 canales latentes.
model = AutoencoderKL(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(128, 256, 512, 512),   # Configuración estándar de MedVAE
    latent_channels=4,          # Verifica si tu versión usa 3 o 4
    num_res_blocks=2,
    attention_levels=(False, False, True, True),
).to(device)

In [ ]:
for i in range(1, 100):
    batch = next(loader)
    inputs = batch["full"].to(device)

    # 1. Obtenemos los parámetros de la distribución (mu y logvar)
    mu, logvar = model.encode(inputs)

    # 2. Realizamos el muestreo (Reparameterization Trick)
    # En MONAI, AutoencoderKL suele tener el método .sampling()
    z = model.sampling(mu, logvar)

    # 3. Decodificamos para obtener la reconstrucción
    reconstruction = model.decode(z)

In [ ]:
with torch.no_grad():
    model.eval()
    reconstruction, mu, logvar = model(inputs)

# --- 5. Visualización de resultados ---
plt.figure(figsize=(15, 5))

# Imagen Original (Full Dose)
plt.subplot(1, 3, 1)
plt.title("Full Dose Original")
plt.imshow(inputs[0, 0].cpu(), cmap="gray")
plt.axis("off")

# Espacio Latente (Visualizando el primer canal de mu)
plt.subplot(1, 3, 2)
plt.title(f"Espacio Latente {mu.shape[2]}x{mu.shape[3]}")
plt.imshow(mu[0, 0].cpu(), cmap="magma")
plt.axis("off")

# Reconstrucción (Sin entrenar se verá ruido o gris)
plt.subplot(1, 3, 3)
plt.title("Reconstrucción VAE")
plt.imshow(reconstruction[0, 0].cpu(), cmap="gray")
plt.axis("off")

plt.tight_layout()
plt.show()

print(f"Dimensiones de entrada: {inputs.shape}")
print(f"Dimensiones latentes:  {mu.shape} (Reducción 8x)")
print(f"Dimensiones reconstrucción: {reconstruction.shape}")